In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 38.8 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from Bio import SeqIO
import pandas as pd

def generate_windows(seq, starts=[0, 256, 483], window_size=512):
    return [str(seq[start:start+window_size]) for start in starts]

records_pos = list(SeqIO.parse("VIS_pos_cd_hit", "fasta"))
records_neg = list(SeqIO.parse("negative_cd_hit_2d.fa", "fasta"))

data = []

# Process positives
for rec in records_pos:
    seq = rec.seq.upper()
    windows = generate_windows(seq)
    for w in windows:
        if len(w) == 512:
            data.append((w, 1))

# Process negatives
for rec in records_neg:
    seq = rec.seq.upper()
    windows = generate_windows(seq)
    for w in windows:
        if len(w) == 512:
            data.append((w, 0))

df = pd.DataFrame(data, columns=["sequence", "label"])
df.to_csv("vis_windows.csv", index=False)


In [ ]:
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer

# Load only the essentials
df = pd.read_csv("vis_windows.csv")
dataset = Dataset.from_pandas(df[["sequence", "label"]].rename(columns={"sequence": "text"}))

# Split before tokenizing
dataset = dataset.train_test_split(test_size=0.2)

# Load DNABERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", trust_remote_code=True)

# Define 6-mer function and tokenize on-the-fly
def kmer_tokenize(example):
    k = 6
    sequence = example["text"]
    kmers = " ".join([sequence[i:i+k] for i in range(len(sequence)-k+1)])
    return tokenizer(kmers, padding="max_length", truncation=True, max_length=512)

# Tokenize with batched mapping (streamed, lower memory)
tokenized = dataset.map(kmer_tokenize, batched=False)


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

In [ ]:
# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df[["kmer_seq", "label"]].rename(columns={"kmer_seq": "text"}))
dataset = dataset.train_test_split(test_size=0.2)

In [ ]:
model_name = "zhihan1996/DNA_bert_6"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=512)

tokenized = dataset.map(tokenize, batched=True)


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    trust_remote_code=True
)


In [ ]:
from transformers import TrainingArguments, Trainer, EvalPrediction
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='binary')
    acc = accuracy_score(p.label_ids, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
trainer.evaluate()
